# Getting Started

**The FunGen-xQTL protocol is a reproducible, end-to-end pipeline for molecular quantitative trait loci (QTL) analysis** — from raw genotypes and phenotypes through discovery, fine-mapping, and integration with GWAS.

This page is a guided on-ramp. A minimal toy dataset of **49 de-identified samples** is used throughout the examples so you can try every pipeline end-to-end before running on real data. In about an hour you'll install the environment, clone the repo, download the demo dataset, and run your first cis-QTL scan.

```{image} images/complete_workflow.png
:alt: FunGen-xQTL analysis workflow
:align: center
:width: 90%
```

:::{seealso}
**New to the consortium?** Start with [How to use the resource](https://statfungen.github.io/xqtl-protocol/README.html#how-to-use-the-resource) on the homepage for the big-picture background, then come back here to set up.
:::


---

## At a Glance

The protocol is modular. Each numbered pipeline is a self-contained [SoS (Script of Scripts)](https://vatlab.github.io/sos-docs/) notebook that can run independently or be chained into the full workflow.

| Stage | What it does | Key pipelines |
|---|---|---|
| **1. Preprocess** | Clean, normalize, and align inputs | phenotype QC, genotype QC, covariate generation |
| **2. Discover** | Scan for QTLs | TensorQTL (cis/trans), APEX (interactions) |
| **3. Fine-map** | Identify credible causal variants | SuSiE, mvSuSiE, fSuSiE |
| **4. Integrate** | Link QTLs to disease and biology | coloc, cTWAS, GWAS integration, enrichment |

Full details with links to every mini-protocol are further down in [Analysis](#analysis). For now, let's get you set up.


---

## Before You Start

You'll need a Linux or macOS shell. Windows users: install [WSL2](https://learn.microsoft.com/windows/wsl/install) first, then follow the Linux path.

| Requirement | Minimum | Recommended |
|---|---|---|
| Disk space | 10 GB (minimal install) | 40 GB (full bioinformatics stack) |
| Memory | 16 GB | 50 GB+ on HPC for the installer |
| Network | GitHub, conda-forge, synapse.org | Same |
| Git | Any recent version | 2.30+ |

:::{tip}
**On HPC, start on a compute node.** The pixi installer is memory-hungry and login nodes will kill it mid-run. Grab an interactive session first:

```bash
srun --mem=50G --pty bash          # SLURM
bsub -Is -M 50000 -n 4 bash        # LSF
```
:::


---

## Step 1. Install pixi

We manage every dependency — Python, R, JupyterLab, common CLI utilities, and a full bioinformatics stack — with [pixi](https://pixi.sh/), a fast reproducible package manager for conda channels. One installer sets it all up.

```bash
curl -fsSL https://raw.githubusercontent.com/StatFunGen/pixi-setup/refs/heads/main/pixi-setup.sh -o pixi-setup.sh
bash pixi-setup.sh
```

The installer will prompt you for two things:

**1. Installation path** — where pixi stores environments and the package cache.

| Setting | When to use |
|---|---|
| `$HOME/.pixi` (default) | Laptops and workstations with plenty of home-directory space |
| `/lab/$USER/.pixi` or scratch | HPC systems with strict home-directory quotas |

**2. Installation type** — pick based on what you plan to do.

| Type | Size | Files | Includes |
|---|---|---|---|
| **1. minimal** | ~5 GB | ~100k | CLI tools, Python data-science stack, JupyterLab, base R (tidyverse, devtools, IRkernel) |
| **2. full** | ~35 GB | ~350k | Everything above, **plus** samtools, bcftools, plink2, GATK4, STAR, Seurat, tensorQTL, Bioconductor |

Choose **minimal** for xQTL runs with pre-processed inputs; choose **full** if you'll also do upstream QC, alignment, or single-cell work.

**Activate and verify:**

```bash
source ~/.bashrc          # or ~/.zshrc on macOS
pixi --version
```

You should see a version number. If not, open a fresh terminal.


---

## Step 2. Add SoS

The protocol's pipelines are written as [SoS](https://vatlab.github.io/sos-docs/) workflows, so we install the SoS suite on top of pixi's Python environment.

```bash
pixi global install --environment python -c conda-forge \
    sos sos-pbs sos-notebook jupyterlab-sos \
    sos-bash sos-python sos-r

pixi run -e python python -m sos_notebook.install
```

**Verify:**

```bash
sos --version
jupyter kernelspec list    # should include 'sos'
```


---

## Step 3. Clone the Protocol

```bash
git clone https://github.com/StatFunGen/xqtl-protocol.git
cd xqtl-protocol
```

:::{note}
**What's in the repo**

| Folder | Contents |
|---|---|
| `pipeline/` | The SoS workflows you'll run |
| `code/` | Notebook-based documentation (this page lives here) |
| `data/` | Small example inputs and configuration templates |
| `website/` | JupyterBook sources for [statfungen.github.io/xqtl-protocol](https://statfungen.github.io/xqtl-protocol/) |
:::


---

## Step 4. Download the Demo Data

Preparation of the demo dataset is documented [on this page](https://github.com/cumc/fungen-xqtl-analysis/tree/main/analysis/Wang_Columbia/ROSMAP/MWE) (a private repository accessible to FunGen-xQTL working group members). The data itself lives on [Synapse](https://www.synapse.org/#!Synapse:syn36416601).

1. Create a free account on [synapse.org](https://www.synapse.org/) — username and password are required to download.
2. Install the Synapse API client into pixi's python environment:
   ```bash
   pixi global install -c conda-forge --environment python synapseclient
   ```
   Alternatively, `pip install synapseclient`. See [the Synapse install docs](https://help.synapse.org/docs/Installing-Synapse-API-Clients.1985249668.html) for details.
3. Every folder at each level of the Synapse project has its own unique ID, so you can download just the subset you need.

**Bulk RNA-seq molecular phenotype quantification** — test data:

```python
import synapseclient
import synapseutils
syn = synapseclient.Synapse()
syn.login("your username on synapse.org", "your password on synapse.org")
files = synapseutils.syncFromSynapse(syn, 'syn53174239', path="./")
```

**xQTL association analysis** — test data:

```python
import synapseclient
import synapseutils
syn = synapseclient.Synapse()
syn.login("your username on synapse.org", "your password on synapse.org")
files = synapseutils.syncFromSynapse(syn, 'syn52369482', path="./")
```


---

## Step 5. Run Your First Workflow

Confirm SoS can see the pipelines:

```bash
sos run pipeline/1_xqtl_association.ipynb -h
```

You should see a list of workflow options. Now run a minimal cis-QTL scan using the demo data you just downloaded:

```bash
sos run pipeline/TensorQTL.ipynb cis \
    --genotype-file   data/example/genotype.bed \
    --phenotype-file  data/example/phenotype.bed.gz \
    --covariate-file  data/example/covariates.tsv \
    --cwd output/demo_tensorqtl
```

Results land in `output/demo_tensorqtl/`. You now have a working environment and a known-good reference run to compare against when you bring in your own data.

:::{tip}
Every pipeline supports `-h` and `--help`, and SoS prints the exact shell commands it runs under the hood — a great way to learn what's happening and to debug failures.
:::


## Analysis

With the environment set up, here's the full protocol in order. Each link is a self-contained mini-protocol; all commands in them should be executed from the command line.

### Molecular Phenotype Quantification

Molecular phenotype data is required to generate QTLs. We support bulk RNA-seq, methylation, and splicing phenotypes. Before quantification, you'll need a handful of [reference data](https://statfungen.github.io/xqtl-protocol/code/reference_data/reference_data.html#) files — reference genomes, gene annotations, variant annotations, linkage disequilibrium maps, and topologically associated domains.

- [Gene expression quantification](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/bulk_expression.html) — RNA-SeQC (gene-level) or RSEM (transcript-level)
- [Alternative splicing quantification](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/splicing.html) — leafcutter2 to identify alternatively excised introns
- [DNA methylation quantification](https://statfungen.github.io/xqtl-protocol/code/molecular_phenotypes/methylation.html) — SeSAMe

Each phenotype then undergoes phenotype-specific QC and normalization.

### Data Pre-Processing

- [Genotype preprocessing](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/genotype_preprocessing.html) — variant filters with bcftools, conversion to plink format, kinship analysis, and genetic PCs on unrelated individuals
- [Phenotype preprocessing](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/phenotype_preprocessing.html) — feature annotation, imputation of missing entries, formatting for QTL analysis
- [Covariate preprocessing](https://statfungen.github.io/xqtl-protocol/code/data_preprocessing/covariate_preprocessing.html) — merge phenotypic data with genetic PCs, then compute hidden factors as additional covariates

### QTL Association Analysis

- [QTL association analysis](https://statfungen.github.io/xqtl-protocol/code/association_scan/qtl_association_testing.html) — TensorQTL scans with cis, trans, and interaction options
- [Hierarchical multiple testing](https://statfungen.github.io/xqtl-protocol/code/association_scan/qtl_association_postprocessing.html) — adjust p-values across levels

### Integrative Analysis

Multiple methods are available for fine-mapping and for linking xQTLs to GWAS and disease biology:

- [TWAS](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/twas_ctwas.html) — identify genes associated with complex traits
- [Univariate fine-mapping and TWAS with SuSiE](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/univariate_fine_mapping_twas_vignette.html) — TWAS weights and credible sets
- [Regression with summary statistics](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/summary_stats_finemapping_vignette.html) — include GWAS summary stats in SuSiE fine-mapping
- [Univariate fine-mapping of functional data](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/univariate_fine_mapping_fsusie_vignette.html) — fSuSiE with epigenomic annotations
- [Colocalization analysis](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/SuSiE_enloc.html) — pairwise colocalization of xQTL and GWAS fine-mapping results
- [Colocboost](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_methods/colocboost.html) — alternative shared-variant discovery across multiple molecular traits
- [Excess-of-overlap enrichment](https://statfungen.github.io/xqtl-protocol/code/enrichment/eoo_enrichment.html) — enrichment of significant variants within genomic annotations
- [Pathway enrichment (GSEA)](https://statfungen.github.io/xqtl-protocol/code/enrichment/gsea.html) — overrepresented pathways in a gene set
- [Stratified LD Score Regression](https://statfungen.github.io/xqtl-protocol/code/enrichment/sldsc_enrichment.html) — heritability partitioning by annotation


---

## Software Environment

Every protocol on this site runs inside the pixi environment configured in Steps 1–2. Once pixi and SoS are installed, each example "just works" — no per-pipeline container, no manual dependency wrangling.

Need something extra? Install it into the right pixi environment:

```bash
# Python package (into the shared python env)
pixi global install -c conda-forge --environment python <package>

# R package (into the r-base env)
pixi global install -c conda-forge --environment r-base r-<package>

# Standalone bioinformatics CLI tool
pixi global install -c bioconda <tool>
```

### Troubleshooting

:::{warning}
**R library conflicts.** If you see an error like

```
Error in dyn.load(file, DLLpath = DLLPath, ...):
unable to load shared object '$PATH/R/x86_64-pc-linux-gnu-library/4.2/stringi/libs/stringi.so':
libicui18n.so.63: cannot open shared object file: No such file or directory
```

your system R libraries are being picked up alongside the pixi ones. Unset them before running the pipeline:

```bash
export R_LIBS=""
export R_LIBS_USER=""
```
:::

**`pixi: command not found`** — open a new terminal, or re-source your shell rc file (`source ~/.bashrc` on Linux/HPC, `source ~/.zshrc` on macOS).

**Installer killed on HPC** — you're on a login node. Request a compute node with ≥ 50 GB memory and re-run.

**`sos: command not found`** — Step 2 didn't complete. Re-run the `pixi global install` command for SoS.

**`ModuleNotFoundError` during a pipeline** — install the missing package into pixi's python env with the command above.

Still stuck? [Open an issue](https://github.com/StatFunGen/xqtl-protocol/issues) with the command you ran and the full error output.


---

## Analyses on High Performance Computing Clusters

The demo on this page runs on a desktop workstation. Production analyses typically run on an HPC cluster, and SoS supports this natively via [SoS Remote Tasks](https://vatlab.github.io/sos-docs/doc/user_guide/task_statement.html) on [configured host computers](https://vatlab.github.io/sos-docs/doc/user_guide/host_setup.html).

We provide a [toy example for running SoS pipelines on a typical HPC cluster environment](https://github.com/statfungen/xqtl-protocol/blob/main/code/misc/Job_Example.ipynb) — first-time users are encouraged to work through it before launching real jobs. It covers the host and task configuration you'll reuse for every subsequent pipeline, and it's schedule-agnostic (SLURM, LSF, SGE, PBS/Torque all work).
